# Model to analyze replacement of Russian imports by LNG

- no network conversion used
- analysis of the impact of LNG import capacity increasement

### Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx
import os
from infrastructure_model_data_import_functions import *
from infrastructure_model_input_data_prep_functions import *
from infrastructure_model_output_data_prep_functions import *

In [2]:
#set name for in- and output
scenario_name = 'ESR_2026_run_2019'

In [3]:
#output specifications
output_file_path = os.path.join('..', '..','01_data', '02_output_data', '02_unidirectional_results','02_paper_ESR', '01_raw_results')
outout_file_name = '\outputs_' + scenario_name + '.xlsx'
#create full ouput paths
output_file_path_excel  = output_file_path + outout_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path_excel))

### Import Data

In [33]:
# Specify the path to your Excel file
input_file_path_data_prep = os.path.join('..', '..','00_code_base', '07_data_prep')
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed', '02_paper_ESR')
excel_file_name = 'inputs_' + scenario_name + '.xlsx'
#data_case_russia_BASE.xlsx
#data_case_russia_peak_hour.xlsx

In [34]:
# Call the function to load the data
df_nodes, df_commodities, df_edges, df_parameter, df_supply_values = load_input_data(input_file_path, excel_file_name)

In [35]:
(network_nodes, commodities, edges, initial_capacities, max_capacities, 
 edge_cost, pipe_new_cost, pipe_conv_cost, pipe_conv_factor, supply_values, 
 node_values) = extract_network_data(
     df_nodes, df_commodities, df_edges, df_parameter, df_supply_values)

### Create input data structure

In [36]:
'''
Create slack nodes for all supply nodes
link a node to supply in case of shortage in the system to all supply nodes
the capacity is infinite but at an infinite (super high) cost
'''
(shortage_list, shortage_edges_list, 
 shortage_capacity_dict, shortage_cost_dict) = create_slack_nodes_and_links(
     node_values, commodities)

In [37]:
'''
Create slack nodes for all supply nodes
enable excess nodes that oversupply is also not a problem
'''
(excess_list, excess_edges_list, 
excess_capacity_dict, excess_cost_dict) = create_excess_nodes_for_supply(
    node_values, commodities)

Check the graph

In [38]:
#check that all nodes are connected via edges
check_graph_connectivity(edges)

The graph is connected.


In [39]:
#check that all nodes are connected via edges and identify not connected components
check_graph_connectivity_and_components(edges)

The graph is connected.


In [40]:
# Check if all nodes are connected
check_if_all_nodes_connected(network_nodes, edges)

All nodes are connected.


## Model

### Create model

In [41]:
# Create a new model
model = gp.Model("Grid_Transformation")

### Define parameters

In [42]:
# Parameters
commodities = commodities  # Commodity types
#real network elements
network_nodes = network_nodes # Nodes of the system
network_edges = edges  # Edges
initial_capacities = initial_capacities # Initial capacities
max_capacities = max_capacities  # Maximum capacities
costs_edge = edge_cost  # Cost to transport from node to node
capacity_new_cost = pipe_new_cost  # Cost to increase capacity
capacity_change_cost = pipe_conv_cost  # Cost to increase capacity
node_value = node_values #contains supply and demand values

#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = pipe_conv_factor

#slack parameters shortage
shortage_nodes = shortage_list
shortage_edges = shortage_edges_list
shortage_capacities = shortage_capacity_dict
shortage_cost = shortage_cost_dict

#slack parameters excess
excess_nodes = excess_list
excess_edges = excess_edges_list
excess_capacities = excess_capacity_dict
excess_cost = excess_cost_dict

#complete network of the model
all_edges = network_edges + excess_edges_list + shortage_edges_list

### Define decision variables

In [43]:
# Decision variables
x_flow = {} #flow of commodity on an edge
x_flow_shortage = {}
x_flow_excess = {}
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    x_flow_shortage[commodity] = {}
    x_flow_excess[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in all_edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_new_cap{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_conv_cap{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"change_{commodity}_{edge[0]}_{edge[1]}")
    for edge in shortage_edges:
        x_flow_shortage[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_shortage_{commodity}_{edge[0]}_{edge[1]}")
    for edge in excess_edges:
        x_flow_excess[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_excess_{commodity}_{edge[0]}_{edge[1]}")
model.update()

In [44]:
network_edges_inc_excess = network_edges + excess_edges_list
network_edges_inc_shortage = network_edges + shortage_edges_list

### Define objective and constraints

In [45]:
# Objective function (minimize total transportation cost + cost to increase and convert capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges) +
    gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges) +
    #gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges),
    gp.quicksum(x_flow_shortage[commodity][edge] * shortage_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in shortage_edges)+
    gp.quicksum(x_flow_excess[commodity][edge] * excess_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in excess_edges),
    GRB.MINIMIZE
)

# Constraints

#inflow of a node must equal the outflow of a node
for node in network_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in network_edges_inc_excess if edge[0] == node)
                                    #substraction of network_edges_inc_shortage not necessary as network_edges_inc_excess
                                    #covers all outgoing flows
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")
#implement excess nodes
for node in excess_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_{commodity}_{node}")

#excess node at the sources to avoid infeasible problems.        
for node in excess_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in excess_edges if edge[1] == node) >= 0,
                        f"supply_excess_{commodity}_{node}")
        
# Add constraint for equality between x_flow and x_flow_excess for the same edge
for commodity in commodities:
    for edge in excess_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_excess[commodity][edge], 
                        f"equality_flow_excess_constraint_{commodity}_{edge}")
        
#Capacity constraint for excess flow
for commodity in commodities:
    for edge in excess_edges:
        model.addConstr(x_flow_excess[commodity][edge] 
                        <= 1000000, 
                        f"capacity_limit_excess{commodity}_{edge[0]}_{edge[1]}")

#implement shortage nodes
for node in shortage_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_shortage_{commodity}_{node}")

#shortage node at the sources to avoid infeasible problems.        
for node in shortage_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in shortage_edges if edge[1] == node) >= 0,
                        f"supply_shortage_{commodity}_{node}")

# Add constraint for equality between x_flow and x_flow_shortage for the same edge
for commodity in commodities:
    for edge in shortage_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_shortage[commodity][edge], 
                        f"equality_flow_shortage_constraint_{commodity}_{edge}")

#Capacity constraint for shortage flow
for commodity in commodities:
    for edge in shortage_edges:
        model.addConstr(x_flow_shortage[commodity][edge] 
                        <= 1000000, 
                        f"capacity_limit_shortage{commodity}_{edge[0]}_{edge[1]}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")


# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraints for edge conversion
for edge in network_edges:
    model.addConstr(Change[commodities[0]][edge] + Change[commodities[1]][edge] == 1, f"switching_constraint_{edge[0]}_{edge[1]}")

for commodity in commodities:
    for edge in network_edges:
        model.addConstr(initial_capacities[commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in all_edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

### Optimize the model

In [46]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4526 rows, 3886 columns and 9098 nonzeros
Model fingerprint: 0x0a437d99
Variable types: 2964 continuous, 922 integer (922 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+09]
  Objective range  [7e+01, 1e+08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+09]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 4435 rows and 3491 columns
Presolve time: 0.01s
Presolved: 91 rows, 395 columns, 696 nonzeros
Variable types: 395 continuous, 0 integer (0 binary)

Root relaxation: objective 8.482217e+11, 178 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl U

### Results processing

In [47]:
#check model status
check_optimization_status(model)

Optimal solution found!


In [48]:
#call the function to store the results into DataFrames
results_df, excess_df, shortage_df = store_optimization_results(
    model,
    commodities,
    network_edges,
    excess_edges,
    shortage_edges,
    x_flow,
    y_new_cap,
    Change,
    z_conv_cap,
    x_flow_excess,
    x_flow_shortage
)

Get results

In [49]:
# Call the function to filter the DataFrames for each commodity
commodity_dfs = filter_commodities_to_dataframe(results_df, commodities)
methane_rows_df = commodity_dfs.get('Methane')

In [50]:
#create a duplicate for further operations
methane_results_df = methane_rows_df

In [51]:
#get initial capacities but in another format than above
initial_capacities_separator_data = create_initial_capacities_separator_dict(df_parameter)
initial_capacities_data = initial_capacities_separator_data

In [52]:
#get the share of use for each infrastructure element
methane_results_df = add_share_column(methane_results_df, initial_capacities_data)
methane_results_df

,Commodity,Edge,Flow,Share
0,Methane,"(BG, RO)",0.000000e+00,0.000000
1,Methane,"(DE, CH)",3.327499e+04,0.279012
2,Methane,"(HU, RS)",2.281376e+04,0.440165
3,Methane,"(SI, HR)",0.000000e+00,0.000000
4,Methane,"(DE, FR)",0.000000e+00,0.000000
...,...,...,...,...
357,Methane,"(IN_Prod, IN)",2.628670e+05,0.000263
358,Methane,"(AS_Prod, AS)",1.599437e+06,0.001599
359,Methane,"(CR_Prod, CR)",1.493733e+06,0.001494
360,Methane,"(EG_Prod, EG)",6.344516e+05,0.000634


In [53]:
#aggregated share of capacity use
# Exclude flows and capacities from the following countries
excluded_countries = ['RU']
result_capacity_wo_ru_df = calculate_aggregated_capacity(methane_results_df, initial_capacities_data, excluded_countries)
result_capacity_wo_ru_df

,Commodity,Type,Total Flow,Total Capacity,Share
0,Methane,LNG,7.301845e+06,1.670024e+11,0.000044
1,Methane,Prod,3.153435e+07,5.300000e+10,0.000595
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [54]:
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows from Russia ('RU')
excluded_countries = ['RU']
result_supply_wo_ru_df = calculate_aggregated_share_supply(methane_results_df, node_values, excluded_countries)
result_supply_wo_ru_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,7.301845e+06,0.000000e+00,0.000000
1,Methane,Prod,3.153435e+07,3.262829e+07,0.966473
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [55]:
search_element = 'ES'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element_and_commodity(results_df, 'Edge', search_element, commodity)
filtered_df

,Commodity,Edge,Flow,New Capacity,Switched,Changed Capacity
10,Methane,"(FR, ES)",0.000000,0.0,1.0,60225.0
22,Methane,"(ES, PT)",0.000000,0.0,1.0,52560.0
37,Methane,"(MA, ES)",114026.898380,0.0,1.0,161622.0
40,Methane,"(DZ, ES)",123041.500000,0.0,1.0,123041.5
46,Methane,"(ES, FR)",0.000000,0.0,1.0,81906.0
52,Methane,"(PT, ES)",0.000000,0.0,1.0,29200.0
114,Methane,"(ES_LNG_imp, ES)",113102.804945,0.0,1.0,587177.0
163,Methane,"(EG_LNG_exp, ES_LNG_imp)",0.000000,0.0,1.0,999999999.0
164,Methane,"(AF_LNG_exp, ES_LNG_imp)",113102.804945,0.0,1.0,999999999.0
207,Methane,"(USA_LNG_exp, ES_LNG_imp)",0.000000,0.0,1.0,999999999.0


In [56]:
search_element = 'RU'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element(methane_results_df, 'Edge', search_element)
filtered_df.head().sum()

Commodity         MethaneMethaneMethaneMethaneMethane
Edge         (RU, UA, LT, RU, RU, LV, RU, DE, RU, TR)
Flow                                   1602026.666598
Share                                        3.106981
dtype: object

In [57]:
#get just excess for methane out of the excess_df
methane_excess_df = excess_df[excess_df['Commodity'].str.contains('Methane', case=False)]
methane_excess_df

,Commodity,Edge,Flow
0,Methane,"(CR_Prod, CR_Prod_excess)",645434.105659
1,Methane,"(DZ_Prod, DZ_Prod_excess)",40597.445827
2,Methane,"(LY_Prod, LY_Prod_excess)",14716.270809
3,Methane,"(AU_Prod, AU_Prod_excess)",393190.779036


In [58]:
'''still not working'''
#check for an unused share of capacities
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows
excluded_countries = ['ES']
methane_excess_share_df = calculate_aggregated_share_supply(methane_excess_df, node_values, excluded_countries)
methane_excess_share_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,0.000000e+00,0.000000e+00,0.000000
1,Methane,Prod,1.093939e+06,3.926109e+07,0.027863
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [59]:
methane_shortage_df = shortage_df[shortage_df['Commodity'].str.contains('Methane', case=False)]
methane_shortage_df

,Commodity,Edge,Flow


In [60]:
# Export to Excel with multiple sheets
with pd.ExcelWriter(full_output_file_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, index=False, sheet_name='raw_results')
    methane_results_df.to_excel(writer, index=False, sheet_name='flows_methane_edges')